## Managed v/s External Tables in Azure Databricks

### Load and Prepare Data

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS data_modelling

In [0]:
import requests
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Define the current catalog and volume
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
volume_base = f"/Volumes/{catalog_name}/default/data_modelling"

# List of files to download
files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

# Define schema for sales data
schema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

# Load data from 2019, 2020, 2021 CSV files
df = spark.read.schema(schema) \
    .option("header", "true") \
    .csv(f'{volume_base}/*_edited.csv')

display(df.limit(10))

### Store Data as Parquet files in External Location

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://unity-catalog-storage-demo@dp750storageaccountdemo.dfs.core.windows.net/sales_data_delta")

### Create an External Table

In [0]:
%sql
CREATE TABLE default.external_sales_table
USING DELTA
LOCATION 'abfss://unity-catalog-storage-demo@dp750storageaccountdemo.dfs.core.windows.net/sales_data_delta';

### Create a Managed Table

In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('default.managed_sales_table')

### Describing the External Table

In [0]:
%sql
DESCRIBE EXTENDED default.external_sales_table

### Describing the Managed Table

In [0]:
%sql
DESCRIBE EXTENDED default.managed_sales_table

### Dropping the Managed Table

In [0]:
%sql
DROP TABLE default.managed_sales_table

### Dropping the External Table

In [0]:
%sql
DROP TABLE default.external_sales_table